`LangGraph`的**底层图算法采用消息传递机制来定义和执行这些图中的交互流程，其中状态（State）组件扮演着关键的载体角色，负责在图的各个节点之间传递信息**。
`LangGraph`框架的核心在于`State`的有效使用和掌握。在复杂的应用中，`State`组件需要存储和管理的信息量会显著增加。核心功能如工具使用、记忆能力和人机交互等，都依赖`State`来实现和维护。

# 1. State的定义模式
<div align=center><img src="./pic/lesson02/1.png" width=100%></div>

- `LangGraph`构建的图中的每个节点都具备访问、读取和写入状态的权限。
- **当某一个节点去修改状态时，它会将此信息广播到图中的所有其他节点**。这种广播机制允许其他节点响应状态的变化并相应地调整其行为。
- 如上图所示，从初始状态（Initial State）开始，其中包含了一条消息 { "x": "10" }，随着消息在节点间通过边传递，每个节点根据其逻辑对状态进行更新。Node 1 和 Node 2 分别对状态进行了处理和变更，结果是在图的末端，我们得到了一个包含三条消息的最终状态 { "x": "10" }, { "x": "11" }, { "y": "9" }。
- **`State`实际上是一个共享的数据结构。状态表现为一个简单的字典。**通过对这个字典进行读写操作，可以实现自左而右的数据流动，从而构建一个可运行的图结构。我们可以利用这个流程来复现并理解图中的动态数据交换，整体的设计如下：
<div align=center><img src="./pic/lesson02/2.png" width=100%></div>

## 1.1 使用字典类型定义状态

In [ ]:
from langgraph.graph import StateGraph

builder = StateGraph(dict)


# node
def add_node(state):
    print(state)
    return {"x": state["x"] + 1}


def sub_node(state):
    print(state)
    return {"y": state["x"] - 2}


builder.add_node("add_node", add_node)
builder.add_node("sub_node", sub_node)

# edge
from langgraph.graph import START, END

builder.add_edge(START, "add_node")
builder.add_edge("add_node", "sub_node")
builder.add_edge("sub_node", END)

graph = builder.compile()

from IPython.display import Image, display

display(Image(graph.get_graph().draw_png()))
print(graph.invoke({"x": 10}))


- 在图的执行过程中，每个节点的函数会被调用，并且接收到前一个节点返回的状态作为输入。
- 每个函数处理完状态后，会输出一个新的状态，传递给下一个节点。
- 这里需要注意的一个关键信息是：节点函数不需要返回整个状态，而是仅返回它们更新的部分。
  - 也就是说：在每个节点的函数内部逻辑中，需要使用和更新哪些`State`中的参数中，只需要在`return`的时候指定即可，不必担心未在当前节点处理的State中的其他值会丢失，因为LangGraph的内部机制已经自动处理了状态的合并和维护

## 1.2 Reducer函数的机制
`LangGraph`内部**原理是：`State`中的每个`key`都有自己独立的`Reducer`函数，通过指定的`reducer`函数应用状态值的更新。**

- `Reducer` 函数用来根据当前的状态（state）和一个操作（action）来计算并返回新的状态。它是一种设计模式，用于将业务逻辑与状态变更解耦，使得状态的变更预测性更强并且容易追踪。
  - 这样的函数通常接收两个参数：当前的状态（state）和一个描述应用了什么操作的对象（action）， 根据 `action` 类型来决定如何修改状态。
  - 比如，在一个购物车应用中，可能会有添加商品、删除商品、修改商品数量等操作。返回一个新的状态对象，而不是修改原始状态对象。**简单理解，`Reducer`函数做的就是根据给定的输入（当前状态和操作）生成新的状态。**

- 增强一
  
```python
class State(TypedDict):
    x: int
    y: int

# 构建图
builder = StateGraph(State) 
```

通过使用 `TypedDict` 来定义 `State` 的模式，从而精确控制图结构中状态信息的格式和类型。
与传统字典类型相比，`TypedDict` 允许明确指定每个键的类型，有助于防止在状态管理过程中出现类型错误。

- 增强二
  
```python
"""
State 的类型是 map
key 是 messages
value 的类型是List[dict], operator.add 表示当有节点返回更新 messages 时，用 + 把新旧列表追加合并
下图中的 List[str] 是错的
"""
class State(TypedDict):
    messages: Annotated[List[dict], operator.add]

# 构建图
builder = StateGraph(State) 
```

`State`还是一个结构化字典，其中包含一个名为 messages 的键，该键保存一个字符串列表。
该`Annotated`包括`operator.add`，表示可以通过使用添加操作将新消息与现有消息组合来更新列表。

- messages：这是字典中的一个键，它表示某个字段（比如消息列表）
- List[str]：这是这个字段值的数据类型，它表示一个字符串类型的列表（即一个包含多个字符串的集合）
- Annotated[List[str], operator.add]：这是一个类型注解的特殊用法，它使用 Annotated 来附加额外的元数据（在这里是 operator.add）
  - Annotated 是 Python 3.9 引入的，它允许你在指定类型的同时附加额外的信息（例如，方法、操作符等）
  - operator.add 是 Python 中的一个加法运算符，通常用来对数字进行加法操作。在这里，它作为元数据附加在 List[str] 类型后面，表明这个字段 messages 可能会涉及到某些与加法相关的操作

<div align=center><img src="./pic/lesson02/5.png" width=100%></div>

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    messages: Annotated[list[str], operator.add]


def add_node(state):
    print(state)
    msg = state["messages"][-1]
    response = {"x": msg["x"] + 1}
    return {"messages": [response]}


def sub_node(state):
    print(state)
    msg = state["messages"][-1]
    response = {"x": msg["x"] - 2}
    return {"messages": [response]}


builder = StateGraph(State)

builder.add_node("add_node", add_node)
builder.add_node("sub_node", sub_node)

# 构建节点之间的边
builder.add_edge(START, "add_node")
builder.add_edge("add_node", "sub_node")
builder.add_edge("sub_node", END)

graph = builder.compile()

from IPython.display import Image, display

display(Image(graph.get_graph().draw_png()))

input_state: State = {"messages": [{"x": 10}]}
graph.invoke(input_state)

operator.add 就是 Reducer函数

这个函数的入参只有两个，任务只有一个： 
合并后的最终值 = Reduce(旧数据, 节点刚返回的新数据)
主要功能是解决当旧数据和新数据同时在手时，规定它们该怎么合并
没有 Reducer 的情况下就是直接覆盖

eg:
```python
def keep_last_3(old_messages: list, new_messages: list):
    # 先拼起来，然后只截取最后 3 条
    return (old_messages + new_messages)[-3:]

class State(TypedDict):
    # 只要有节点返回新消息，自动只留最后 3 条！
    messages: Annotated[list, keep_last_3]
```

## 1.3 在图状态中处理消息的思路

In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, END

import operator
from typing import Annotated, TypedDict

qwen_llm = ChatOpenAI(
    model="qwen3.7-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


class State(TypedDict):
    messages: Annotated[list[dict], operator.add]


def chat_with_model(state):
    print(state)
    messages = state["messages"]
    response = qwen_llm.invoke(messages)
    return {"messages": [response]}


def convert_messages(state):
    # "您是一位数据提取专家，负责从文本中检索关键信息。请为所提供的文本提取相关信息，并以 JSON 格式输出。概述所提取的关键数据点。"
    EXTRACTION_PROMPT = """
    You are a data extraction specialist tasked with retrieving key information from a text.
    Extract such information for the provided text and output it in JSON format. Outline the key data points extracted.
    """
    print(state)
    messages = [
        SystemMessage(EXTRACTION_PROMPT),
        HumanMessage(content=state["messages"][-1].content),
    ]
    response = qwen_llm.invoke(messages)
    return {"messages": [response]}


builder = StateGraph(State)

builder.add_node("chat_with_model", chat_with_model)
builder.add_node("convert_messages", convert_messages)

builder.set_entry_point("chat_with_model")
builder.add_edge("chat_with_model", "convert_messages")
builder.add_edge("convert_messages", END)

graph = builder.compile()
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

input_message = {"messages": [HumanMessage(content="介绍下你自己")]}
response = graph.invoke(input_message)

print(response)

print(response["messages"][-1].content)


## 1.4 MessageGraph源码功能解析(已弃用)

```
The class "MessageGraph" is deprecated
  MessageGraph is deprecated in langgraph 1.0.0, to be removed in 2.0.0. Please use StateGraph with a `messages` key instead.Pylance
```

如果想手动更新图形状态中的消息（例如人机交互），**`operator.add`仅仅能做到讲发送到 Graph 的手动状态更新附加到现有的消息列表中，而不是更新现有的消息**

为此，我们则需要一个可以跟踪消息 ID 并覆盖现有消息（如果更新）的`Reducer`函数

`LangGraph`预构建的`add_messages`是更高级的`Reducer`, **对于全新的消息，它会附加到现有列表，但它也会正确处理现有消息的更新**

---

`MessageGraph` 是 `StateGraph` 的一个子类，使用了 `Annotated[list[AnyMessage], add_messages]` 来初始化其基类 `StateGraph`

`MessageGraph` 中的每个节点都将消息列表作为输入，并返回零个或多个消息作为输出。`add_messages`函数用于将每个节点的输出消息合并进图的状态中已存在的消息列表。其源码定义如下：

https://github.com/langchain-ai/langgraph/blob/e3ef9adac7395e5c0943c22bbc8a4a856b103aa3/libs/langgraph/langgraph/graph/message.py#L150

```python
class MessageGraph(StateGraph):
    def __init__(self) -> None:
        super().__init__(Annotated[list[AnyMessage], add_messages]])
```

## 1.5 add_messages

现代写法

```python
from langgraph.graph import StateGraph, MessagesState

# 1. 直接使用官方内置的 MessagesState
# 它内部自带了 messages 字段和自动追加的 reducer！
builder = StateGraph(MessagesState)

# 2. 节点逻辑（注意：入参和返回值都是字典）
def chatbot(state: MessagesState):
    latest_msg = state["messages"][-1]
    # ...
    return {"messages": [response]}
```

更高级的写法

如果你不仅要存消息，还想存自己的自定义字段

```python
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, add_messages
from langchain_core.messages import BaseMessage

# 自定义 State：既有消息列表，又有你自己的业务数据
class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    user_name: str    # 你的自定义字段
    step_count: int   # 你的自定义计数器

builder = StateGraph(State)
```

### add_messages 源码

```python

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


def add_messages(left: Messages, right: Messages) -> Messages:
    """Merges two lists of messages, updating existing messages by ID.

    By default, this ensures the state is "append-only", unless the
    new message has the same ID as an existing message.

    Args:
        left: The base list of messages.
        right: The list of messages (or single message) to merge
            into the base list.

    Returns:
        A new list of messages with the messages from `right` merged into `left`.
        If a message in `right` has the same ID as a message in `left`, the
        message from `right` will replace the messagage(content='Hello', id=...)]}
        ```

    """
    # coerce to list
    if not isinstance(left, list):
        left = [left]  # type: ignore[assignment]
    if not isinstance(right, list):
        right = [right]  # type: ignore[assignment]
    # coerce to message
    left = [
        message_chunk_to_message(cast(BaseMessageChunk, m))
        for m in convert_to_messages(left)
    ]
    right = [
        message_chunk_to_message(cast(BaseMessageChunk, m))
        for m in convert_to_messages(right)
    ]
    # assign missing ids
    for m in left:
        if m.id is None:
            m.id = str(uuid.uuid4())
    for m in right:
        if m.id is None:
            m.id = str(uuid.uuid4())
    # merge
    left_idx_by_id = {m.id: i for i, m in enumerate(left)}
    merged = left.copy()
    ids_to_remove = set()
    for m in right:
        if (existing_idx := left_idx_by_id.get(m.id)) is not None:
            if isinstance(m, RemoveMessage):
                ids_to_remove.add(m.id)
            else:
                merged[existing_idx] = m
        else:
            if isinstance(m, RemoveMessage):
                raise ValueError(
                    f"Attempting to delete a message with an ID that doesn't exist ('{m.id}')"
                )

            merged.append(m)
    merged = [m for m in merged if m.id not in ids_to_remove]
    return merged
```

- left （ Messages ） – 消息的基本列表
- right （ Messages ） – 要合并到基本列表中的消息列表（或单个消息）
  
`add_messages`整体的核心逻辑是合并两个消息列表，按 `ID` 更新现有消息。默认情况下，状态为“仅附加”

若 right 的消息与 left 的消息具有相同的 ID，则right的消息将替换left的消息

In [ ]:
from langgraph.graph import StateGraph, add_messages

msgs1 = [HumanMessage(content="你好, xxx。", id="1")]
msgs2 = [HumanMessage(content="你好。", id="1")]

add_messages(msgs1, msgs2)

In [12]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
import os
from langchain_core.messages import AIMessage
from IPython.display import Image, display


builder = StateGraph(MessagesState)

qwen_llm = ChatOpenAI(
    model="qwen3.7-plus",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


def chat_bot(state: MessagesState):
    return {"messages": [qwen_llm.invoke(state["messages"])]}


builder.add_node("chat_bot", chat_bot)

builder.add_edge(START, "chat_bot")
builder.add_edge("chat_bot", END)

graph = builder.compile()

# display(Image(graph.get_graph(xray=True).draw_mermaid_png()))


def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [("user", user_input)]}):
        for value in event.values():
            print(f"Qwen: {value['messages'][-1].content}")


while True:
    try:
        user_input = input("用户提问: ")
        if user_input.lower() in ["退出", "再见", "quit", "q"]:
            print("再见")
            break
        print(f"User: {user_input}")
        stream_graph_updates(user_input)
    except Exception as e:
        print(f"发生错误: {e}")

User:你好
Qwen: 你好！有什么我可以帮你的吗？
User:普通的程序员预计多久会被完全取代
Qwen: 要回答“普通的程序员预计多久会被完全取代”这个问题，首先需要明确两个概念：**什么是“完全取代”**，以及**什么是“普通的程序员”**。

如果“完全取代”指的是 **“软件公司不再需要雇佣人类来参与任何软件开发环节”**，那么在可预见的未来（至少10-20年内），这**几乎不可能发生**。
但如果“完全取代”指的是 **“不再需要大量从事基础代码编写、CRUD（增删改查）的初级/中级码农”**，那么这个过程**已经开始，并将在未来3-5年内加速完成**。

业界的一个共识是：**取代你的不是AI，而是“熟练使用AI的程序员”。**

我们可以从时间线、AI的局限性（人类的护城河）以及行业演变三个维度来深度剖析这个问题。

---

### 一、 时间线推演：程序员的演变史

#### 1. 短期（现在 - 3年）：AI作为“超级外挂”，淘汰“代码搬运工”
*   **现状**：Copilot（如Cursor、GitHub Copilot）已经能完成60%以上的重复性代码编写、单元测试生成和基础Bug排查。
*   **影响**：纯粹的“代码搬运工”（只负责把产品经理的需求翻译成基础代码，不思考业务逻辑的人）需求将大幅锐减。初级岗位的门槛被无限拔高，以前需要3个初级程序员干的活，现在1个中级程序员+AI就能搞定。

#### 2. 中期（3 - 5年）：AI Agent成熟，“一人团队”成为常态
*   **趋势**：AI将从“补全工具”进化为“自主Agent”。你只需要输入一段自然语言需求，AI Agent能自动完成需求拆解、数据库设计、前后端代码生成、自动化测试并部署。
*   **影响**：“普通程序员”的定义被改写。程序员不再需要记忆API和语法，而是变成 **“代码审查员”** 和 **“需求翻译官”**。团队规模会急剧缩小，一个懂业务、懂架构的资深工程师带着几个AI Agent，就能完成过去一个10人团队的产出。

#### 3. 长期（5 - 10年以上）：软件工程范式转移
*   **趋势**：自然语言可能成为新的“编程语言”。软件开发的重心从“如何实现（How）”彻底转移到“要做什么（What）”和“为什么做（Why）”。
*   *

```python
event = {
    "刚跑完的节点名称": 节点 return 的增量字典
}

"""
例如
"""
event = {
    "chatbot": {
        "messages": [
            AIMessage(content="你好！有什么我可以帮你的？")
        ]
    }
}
```